# 🎬 YouTube Átirat Letöltő (YouTube Transcript Downloader Pro)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krisatrader/YT-text-downloader/blob/main/youtube_transcript_downloader_colab.ipynb)

Tölts le teljes YouTube lejátszási listákat, csatornákat vagy videókat betűről betűre **.md** és **.txt** formátumban teljesen ingyen, közvetlenül a Google szupergyors felhőjéből!

---

In [ ]:
#@title 🚀 1. Telepítés és Előkészítés (Kattints ide a futtatáshoz ▶️)
#@markdown Ez a lépés letölti a kódot a GitHub-ról és telepíti a szükséges csomagokat.

import os, sys

!rm -rf /content/YT-text-downloader
!git clone https://github.com/krisatrader/YT-text-downloader.git /content/YT-text-downloader
%cd /content/YT-text-downloader
!pip install -q -r requirements.txt

print("\n✅ Sikeres előkészítés! Válassz az alábbi 2/A vagy 2/B futtatási mód közül.")

In [ ]:
#@title 🌐 2/A. Webes Kezelőfelület Indítása (Publikus Linken)
#@markdown Indítsd el a teljes webes felületet valós idejű folyamatkövetéssel, előnézettel és beépített Sütikezelővel.

import os, sys, time, subprocess, re, urllib.request

%cd /content/YT-text-downloader

# Cloudflared letöltése az ingyenes és biztonságos publikus aldomainhez
cf_path = "/usr/local/bin/cloudflared"
if not os.path.exists(cf_path):
    print("⬇️ Cloudflare Tunnel komponens letöltése...")
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf_path)
    os.chmod(cf_path, 0o777)

# Webes FastAPI kiszolgáló indítása a háttérben
print("🚀 Web szerver indítása...")
web_proc = subprocess.Popen([sys.executable, "web_app.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2.5)

# Publikus tunnel nyitása
print("🔗 Publikus webcím generálása...")
tunnel_proc = subprocess.Popen([cf_path, "tunnel", "--url", "http://127.0.0.1:8000"], stderr=subprocess.PIPE, text=True)

tunnel_url = None
for line in tunnel_proc.stderr:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        print("\n" + "="*65)
        print(f"🎉 A WEB FELÜLET SIKERESEN ELINDULT!")
        print(f"👉 KATTINTS IDE A MEGNYITÁSHOZ: {tunnel_url}")
        print("="*65 + "\n")
        break

if not tunnel_url:
    print("⚠️ Nem sikerült generálni a publikus linket, próbáld újra!")

In [ ]:
#@title ⚡ 2/B. Gyors Letöltés Közvetlenül a Colab-ban (Űrlap + Automatikus ZIP Letöltés)
#@markdown Töltsd ki a mezőket és kattints a futtatás gombra! A folyamat végén a ZIP automatikusan letöltődik a gépedre.

YouTube_URL = "https://youtube.com/playlist?list=PLqRSM1XGfwbNZvCJ4Pz044LYPgkdUvi78" #@param {type:"string"}
Formatum = "both" #@param ["both", "md", "txt"]
Nyelvek = "hu,en" #@param {type:"string"}
Idobelyegek = True #@param {type:"boolean"}
Max_Videok = 0 #@param {type:"integer"}
Keses_Masodperc = "2.0-3.0" #@param {type:"string"}
#@markdown **Opcionális Sütik (ha a YouTube HTTP 429-cel korlátoz, illeszd be ide a cookies.txt tartalmát):**
Cookies_Szoveg = "" #@param {type:"string"}

import os, sys, zipfile
from google.colab import files

%cd /content/YT-text-downloader
from transcript_downloader import DownloaderEngine, CookieManager, sanitize_filename

if Cookies_Szoveg.strip():
    cm = CookieManager("transcripts_output/.saved_cookies.txt")
    cm.save_cookie_text(Cookies_Szoveg.strip())
    print("🍪 Sütik sikeresen beállítva!")

# Késleltetés feldolgozása
delay_min, delay_max = 2.0, 3.0
if "-" in Keses_Masodperc:
    try:
        dparts = Keses_Masodperc.split("-")
        delay_min, delay_max = float(dparts[0]), float(dparts[1])
    except Exception:
        pass

engine = DownloaderEngine(
    output_dir="transcripts_output",
    output_format=Formatum,
    delay_range=(delay_min, delay_max),
    preferred_languages=[l.strip() for l in Nyelvek.split(",") if l.strip()],
    include_timestamps=Idobelyegek,
    limit=Max_Videok if Max_Videok > 0 else None
)

print(f"🚀 Letöltés indítása: {YouTube_URL}...")
res = engine.run(YouTube_URL)

if res and res.get("collection_title"):
    col_title = res["collection_title"]
    clean_title = sanitize_filename(col_title)
    target_dir = os.path.join("transcripts_output", clean_title)
    zip_filename = f"{clean_title}.zip"

    with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, filenames in os.walk(target_dir):
            for fname in filenames:
                fpath = os.path.join(root, fname)
                zf.write(fpath, os.path.relpath(fpath, target_dir))

    print(f"\n📦 ZIP fájl elkészült: {zip_filename}")
    print("⬇️ Letöltés kezdeményezése a böngésződben...")
    files.download(zip_filename)
else:
    print("⚠️ Nem sikerült a gyűjtemény letöltése vagy nincsenek videók.")